# Final Review Session — Detailed Notes (Sessions 9–21)
**Course:** CS490/5590 — Quantum Computing Applications in Data Science, AI, & Deep Learning  
**Instructor:** Luke Miller

> **Goal.** One-stop, student-facing study guide for the final: concise concept maps, “what to know” checklists, pitfalls + fixes, minimal math you must remember, and a compact practice set with outline solutions.

---

## 0) Scope & What’s Fair Game

### Sessions in scope
- **9** VQE (variational ground-state solver)
- **10** Quantum Kernels & **QSVMs**
- **11** Quantum Feature Selection (MI/kernels)
- **14** **QNNs** (variational classifiers/regressors)
- **15** **Hybrid Methods** (Qiskit ⇄ PyTorch/TF)
- **16** **QCNNs** (quanvolution + pooling)
- **17** **QGNNs** (graph encoding, message passing)
- **19** Quantum **Bayes**
- **20** Quantum **K-Means** (distance circuits)
- **21** Quantum **GANs** (generator/discriminator)

**Final emphasis:** theory, circuit design, where quantum helps (and where it doesn’t), NISQ-aware reasoning.

---

## 1) Variational Methods (VQE & friends)

### Key ideas
- **Ansatz** $U(\theta)$, **observable** $H=\sum_i h_i P_i$ (Pauli strings).  
- **Cost:** $\langle H\rangle_\theta = \sum_i h_i \langle P_i\rangle_\theta$.  
- **Loop:** prepare → measure → classical optimizer updates $\theta$.

### Must-know
- **Why variational?** Shallow circuits; noise-tolerant(ish).  
- **Optimizers:** SPSA (noise robust), COBYLA (derivative-free).  
- **Barren plateaus:** deep/random ansätze ⇒ tiny gradients.

### Mini-formulas
- **Parameter-shift:** $\partial_{\theta}\langle O\rangle = \tfrac12(\langle O\rangle_{\theta+\pi/2}-\langle O\rangle_{\theta-\pi/2})$.

### Pitfalls → Fix
- Too-deep ansatz → start with **2-local**, reps=1; grow.  
- Measuring every Pauli separately → **group commuting** terms.

---

## 2) Quantum Kernels & QSVMs

### Feature maps (examples)
- **ZZFeatureMap:** data-encoded Z rotations + entangling $e^{-i\phi_{ij} Z_i Z_j}$.  
- **Kernel:** $K(x,z)=|\langle 0|U_\phi^\dagger(x)U_\phi(z)|0\rangle|^2$.

### Must-know
- Why it can help: **implicit high-dim Hilbert space** with **entanglement**.  
- Costs: $O(N^2)$ kernel evals + **shot noise**.

### Pitfalls → Fix
- Wrong angle scaling → normalize features to $[-\pi,\pi]$.  
- Ignoring shot/readout error → use **readout mitigation**; average shots.

---

## 3) Quantum Feature Selection

### Two routes
- **Kernel-based relevance:** rank features by kernel alignment/accuracy deltas.  
- **Quantum MI (conceptual):** $I(X;Y)=S(\rho_X)+S(\rho_Y)-S(\rho_{XY})$ (estimate overlaps/entropies).

### Pitfalls → Fix
- Unstable MI under noise → prefer **kernel scores** + classical wrappers (RFE/LASSO).  
- Too many qubits for features → **subset/ego** strategies, batching.

---

## 4) QNNs, Hybrid Methods, QCNNs, QGNNs

### QNN (classifier/regressor)
- **Pipeline:** encode $x$ → ansatz $U(\theta)$ → measure expectations → loss (BCE/MSE) → optimizer (Adam/SPSA).
- **Barren plateaus:** mitigate via **identity init**, **problem-aware entanglement**, shallow depth.

### Hybrid (Qiskit + PyTorch/TF)
- QNN as a **layer**; backprop uses **parameter-shift** or **SPSA**.  
- Data **encoding/decoding** is the bandwidth bottleneck—keep simple (angle encoding).

### QCNN
- **Quanvolution:** same PQC (shared parameters) per **patch**; pooling by measurement/discard.  
- Use for **small images** (e.g., 2×2/4×4 patches) + shallow depth.

### QGNN
- **Graph encoding:** angle features per node; **ego-subgraphs** for NISQ.  
- **Message passing:** shared PQCs over node+neighbors; aggregate classically.

---

## 5) Advanced QML: Bayes, K-Means, GANs

### Quantum Bayes
- **Quantum Bayesian networks:** nodes as states, edges as controlled unitaries.  
- **Phase estimation idea:** encode probabilities/likelihood phases (conceptual).  
- **Use:** uncertainty quantification; heavy for NISQ → small demos only.

### Quantum K-Means
- **Distances:** **swap test** for overlaps $|\langle x|c_k\rangle|^2$ ⇒ cosine similarity; convert to Euclidean for normalized vectors.  
- **Hybrid loop:** quantum distances, classical updates.

### QGAN
- **Generator:** PQC sampling bitstrings; **Discriminator:** classical (practical) or PQC.  
- **Losses:** BCE or Wasserstein(+GP).  
- **Stability:** shallow circuits, label smoothing, SPSA/Adam, entropy regularization.

---

## 6) Common Pitfalls & Quick Fixes

| Pitfall | Why it hurts | Fix |
|---|---|---|
| Over-deep ansatz | plateaus + noise | reps=1, identity init, grow later |
| Bad feature scaling | kernel/QNN inconsistent | scale to $[-\pi,\pi]$ or z-score + gain |
| Shot starvation | noisy expectations | ≥ 512–1024 shots; average; use mitigation |
| Oracle-like assumptions | unrealistic speedup claims | state explicit **assumptions**; compare to classical baselines |
| Ignoring hardware topology | extra SWAPs/depth | transpile w/ layout; native gate set |
| No classical baseline | unclear “advantage” | compare to RBF-SVM / small CNN / GNN |

---

## 7) Minimal Patterns (Qiskit pseudo-blueprints)

### VQE expectation (concept)
```python
# given ansatz(theta), PauliList H = sum_i h_i P_i
for step in range(T):
    # prepare state
    # measure grouped commuting Paulis -> expectations
    # E = sum_i h_i * <P_i>
    # optimizer.update(theta, grad or SPSA)
```

### Kernel evaluation
```python
# circuit: 0 -- U_phi(x_j) -- U_phi^\dagger(x_i) -- measure
# prob(|0..0>) ≈ inner product squared = K(x_i, x_j)
```

### Swap test distance
```python
# ancilla H; CSWAP over |x>, |c>; ancilla H; measure ancilla
# p(anc=0) = (1 + |<x|c>|^2)/2
```

### QNN training loop
```python
for epoch in range(E):
    # encode x -> U_phi(x)
    # apply ansatz U(theta)
    # measure Z-expectations -> logits
    # loss = BCE/MSE
    # update theta (Adam or SPSA/param-shift)
```

---

## 8) Practice Set (with outline solutions)

### P1. **VQE (2 qubits)**  
**H** = $Z_0Z_1 + X_0$. Choose an ansatz, write $\langle H\rangle$, optimization plan.  
**Sketch:** Use $U(\theta)=\text{CNOT}(0,1)\,[R_y(\theta_0)\otimes R_y(\theta_1)]$.  
Measure $\langle Z\otimes Z\rangle$ and $\langle X\otimes I\rangle$. Optimize via **SPSA**.

---

### P2. **ZZ kernel**  
Given $U_\phi(x)=\prod_i R_Z(x_i)\prod_{i<j} e^{-i \alpha_{ij} Z_iZ_j}$.  
**Task:** Circuit for $K(x,z)$.  
**Sketch:** Prepare $|0\rangle$, apply $U_\phi(z)$ then $U_\phi^\dagger(x)$, measure $P_{0^{\otimes n}}$. Entanglers create cross-terms improving separability.

---

### P3. **Feature selection (3 features)**  
**Plan:** For each feature $x_k$: build a 1–2 qubit feature map with $y$, compute a relevance score (kernel alignment or MI proxy using overlaps), rank features, pick top-k.  
**Noise note:** average shots; prefer kernel-based scores on NISQ.

---

### P4. **QCNN patch**  
Design a 4-qubit quanvolution block for a $2\times2$ patch.  
**Sketch:** Encode pixels via $R_y(\beta p_i)$. Block: `ry(θ)`, ring `cx`, optional `rz`, measure $\langle Z\rangle$. **Share parameters** across all patches.

---

### P5. **QK-Means distance**  
Show how swap test yields cosine similarity for normalized $x,c$.  
**Sketch:** $||x-c||^2 = 2(1-\langle x|c\rangle)$. Swap test gives $|\langle x|c\rangle|^2$; for nonnegative overlaps use $\sqrt{\cdot}$ or estimate sign if needed; or use direct inner-product circuits.

---

### P6. **QGAN loop**  
Write losses and steps for QG (quantum) + CD (classical).  
**Sketch:**  
- Train D: $\mathcal{L}_D=\text{BCE}(D(x),1)+\text{BCE}(D(\tilde{x}),0)$.  
- Train G: $\mathcal{L}_G=\text{BCE}(D(\tilde{x}),1)$.  
- $\tilde{x}$ from PQC sampler; shots 1024; label smoothing 0.9.

---

### P7. **Anomaly via QAE**  
Explain using **trash-state loss**.  
**Sketch:** Train encoder to push trash to $|00..\rangle$. At test, **low** $P(\text{trash}=0)$ ⇒ **high** reconstruction error ⇒ anomaly.

---

### P8. **Hybrid gradient**  
Explain parameter-shift in a Torch loop.  
**Sketch:** Forward: expectations → loss. Backward: two extra forward passes per parameter with $\pm\pi/2$ shifts (or SPSA).

---

## 9) Exam Logistics

**Duration:** 2 hours  
**Format:**  
- Part A (MCQ, ~20%) — short theory & definitions  
- Part B (Short answers, ~30%) — circuit sketches, explain speedups/limits  
- Part C (Design, ~50%) — build a small QML pipeline: map data → circuit → measurements → cost; reason about noise & baselines

**Grading tips:** Show **assumptions**, report **shots/depth**, include **classical baseline** in comparisons.

---

## 10) One-Page Checklists

### Must-remember formulas
- Param-shift: $\partial_\theta f = \frac12[f(\theta+\pi/2)-f(\theta-\pi/2)]$.  
- ZZ kernel: $K(x,z)=|\langle 0|U^\dagger_\phi(x)U_\phi(z)|0\rangle|^2$.  
- Swap test: $p(0)=(1+|\langle \psi|\phi\rangle|^2)/2$.  
- Grover-style counting (contextual to K-Means nearest-centroid reasoning): iterations scale as $\sqrt{N}$ (reference only).

### Noise-aware habits
- Use transpiler **level ≥ 2**, noise-adaptive layout.  
- Readout mitigation; batch shots.  
- Keep reps low; identity init.

---

## 11) 5-Day Crunch Plan (optional)

- **D-5:** Rework VQE & kernel derivations; code a 2-qubit VQE.  
- **D-4:** QNN classification on toy data; param-shift vs SPSA.  
- **D-3:** QCNN patch + swap test function; tiny K-Means loop.  
- **D-2:** QGAN sampler with classical D; monitor mode coverage.  
- **D-1:** Mixed practice set; write **one sheet** of formulas; sleep.

---

## 12) Quick Answers (selected from practice)

- **P1 noise impact:** Biases $\langle P_i\rangle$, increases variance; mitigate via readout calibration, more shots, grouping, shorter depth.  
- **P2 entanglement & separability:** Entanglers create higher-order feature interactions (nonlinear decision boundaries).  
- **P3 noise & MI:** Entropy/overlap estimates are shot-sensitive; prefer kernel alignment + cross-validation.  
- **P5 encoding effect:** Amplitude encoding reduces qubits but is deep; angle encoding shallow but needs more qubits.  
- **P6 barren plateaus:** Use shallow G/D, identity init, label smoothing, WGAN-GP if classical D.

---

## 13) TL;DR
- **Variational = NISQ-friendly.** Keep circuits **shallow**, init near **identity**, choose **robust** optimizers.  
- **Always compare** to strong **classical baselines** and report **resources** (shots/depth).  
- **Kernels/QNNs/QCNNs/QGNNs**: get encoding right; entanglement is a tool, not a guarantee.  
- **Generative & Bayes**: great concepts; scale cautiously on NISQ.

Good luck—you’ve got this. 👊
